# Demo — Notebooks Databricks

Un notebook debe ser legible como documento técnico, no solo como una secuencia de consultas.

En Databricks, una celda Markdown permite documentar:

- El objetivo de la demo.
- El contexto de una tabla o dataset.
- Las decisiones técnicas.
- Las validaciones esperadas.
- Las conclusiones del análisis.

## Caso de esta demo

Partimos de una tabla Delta creada desde un fichero CSV de pedidos.

| Elemento | Valor |
|---|---|
| Dataset original | `orders.csv` |
| Tabla Delta | `training.lesson_01.orders_delta_demo` |
| Lenguaje de consulta | SQL |
| Objetivo | Explorar, validar y consultar una tabla Delta |

## Estructura recomendada del notebook

1. Contexto del ejercicio.
2. Exploración inicial de la tabla.
3. Validación de número de filas.
4. Consultas de análisis.
5. Revisión de metadatos.
6. Conclusión.


# Ejemplo práctico de Markdown

Este bloque muestra los elementos básicos que después usarán los alumnos en la práctica.

## Títulos

Usamos `#`, `##` y `###` para separar secciones.

## Negrita y código inline

La tabla principal de la demo es **`training.lesson_01.orders_delta_demo`**.

## Tabla Markdown

| Validación | Resultado esperado |
|---|---|
| La tabla existe | Sí |
| Número de filas | 100 |
| Columnas principales | 7 |
| Consulta SQL básica | Ejecuta sin error |

## Bloque de código documentado

```sql
SELECT
    COUNT(*) AS total_rows
FROM training.lesson_01.orders_delta_demo;
```

## Checklist Markdown

- [ ] La tabla se puede consultar.
- [ ] El conteo devuelve 100 filas.
- [ ] Las columnas esperadas existen.
- [ ] Las consultas de negocio devuelven resultados.


## 1. Exploración inicial

Objetivo: mostrar una muestra de los registros cargados en la tabla Delta.

Esta consulta equivale conceptualmente a un `SELECT TOP 10` en SQL Server.


In [0]:
%sql
SELECT
    *
FROM training.lesson_01.orders_delta_demo
LIMIT 10;

## 2. Validación de volumen

Objetivo: comprobar que la tabla contiene el número esperado de registros.

Para este dataset, el resultado esperado es **100 filas**.


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows
FROM training.lesson_01.orders_delta_demo;

## 3. Análisis agregado por país y canal

Objetivo: calcular pedidos e importe total por combinación de país y canal de venta.

Esta consulta introduce los patrones SQL básicos que se usarán en la práctica: `GROUP BY`, `COUNT`, `SUM`, `ROUND` y `ORDER BY`.


In [0]:
%sql
SELECT
    country,
    channel,
    COUNT(*) AS total_orders,
    ROUND(SUM(amount), 2) AS total_amount
FROM training.lesson_01.orders_delta_demo
GROUP BY
    country,
    channel
ORDER BY
    total_amount DESC;

## 4. Estructura de la tabla

Objetivo: revisar columnas y tipos de datos registrados en la tabla.

Esta validación permite comprobar si la carga desde CSV ha interpretado correctamente el esquema.


In [0]:
DESCRIBE TABLE training.lesson_01.orders_delta_demo;

## 5. Historial Delta

Objetivo: mostrar el historial de operaciones de la tabla Delta.

Esta consulta ayuda a conectar la práctica con el concepto de `_delta_log`, commits e historial transaccional.


In [0]:
DESCRIBE HISTORY training.lesson_01.orders_delta_demo;

# Bloque adicional — Widgets y consultas parametrizadas

Los widgets permiten ejecutar el mismo notebook con distintos valores sin modificar el SQL.

En esta demo se van a crear cuatro parámetros:

| Widget | Uso |
|---|---|
| `table_name` | Nombre completo de la tabla que se va a consultar |
| `selected_country` | País usado como filtro |
| `selected_status` | Estado del pedido usado como filtro |
| `minimum_amount` | Importe mínimo del pedido |

La idea operativa es pasar de consultas rígidas a consultas reutilizables:

```sql
WHERE country = 'Spain'
```

a:

```sql
WHERE country = :selected_country
```


In [0]:
%sql
CREATE WIDGET TEXT table_name DEFAULT "training.lesson_01.orders_delta_demo";


In [0]:
%sql
CREATE WIDGET DROPDOWN selected_country DEFAULT "Spain" CHOICES SELECT * FROM (
    VALUES
        ("Spain"),
        ("France"),
        ("Germany"),
        ("Italy"),
        ("Netherlands"),
        ("Portugal")
);


In [0]:
%sql
CREATE WIDGET DROPDOWN selected_status DEFAULT "Completed" CHOICES SELECT * FROM (
    VALUES
        ("Completed"),
        ("Cancelled"),
        ("Pending")
);


In [0]:
%sql
CREATE WIDGET TEXT minimum_amount DEFAULT "0";


## Consulta parametrizada 1 — Pedidos por país y estado

Esta consulta usa los widgets `selected_country`, `selected_status` y `minimum_amount`.

Cambia los valores de los widgets en la parte superior del notebook y vuelve a ejecutar la celda.


In [0]:
%sql
SELECT
    country,
    status,
    COUNT(*) AS total_orders,
    ROUND(SUM(amount), 2) AS total_amount,
    ROUND(AVG(amount), 2) AS average_amount
FROM IDENTIFIER(:table_name)
WHERE
    country = :selected_country
    AND status = :selected_status
    AND amount >= CAST(:minimum_amount AS DOUBLE)
GROUP BY
    country,
    status
ORDER BY
    total_amount DESC;


## Consulta parametrizada 2 — Ventas por canal para el país seleccionado

Esta consulta mantiene el filtro de país, pero cambia la agregación para analizar canales.

El mismo widget se reutiliza en otra pregunta de negocio.


In [0]:
%sql
SELECT
    channel,
    COUNT(*) AS total_orders,
    ROUND(SUM(amount), 2) AS total_amount,
    ROUND(AVG(amount), 2) AS average_order_amount
FROM IDENTIFIER(:table_name)
WHERE
    country = :selected_country
    AND amount >= CAST(:minimum_amount AS DOUBLE)
GROUP BY
    channel
ORDER BY
    total_amount DESC;


## Consulta parametrizada 3 — Evolución diaria con filtros

Esta consulta permite revisar las ventas por día para el país y estado seleccionados.

El patrón será útil más adelante para notebooks operativos y jobs parametrizados.


In [0]:
%sql
SELECT
    order_date,
    COUNT(*) AS total_orders,
    ROUND(SUM(amount), 2) AS total_amount
FROM IDENTIFIER(:table_name)
WHERE
    country = :selected_country
    AND status = :selected_status
    AND amount >= CAST(:minimum_amount AS DOUBLE)
GROUP BY
    order_date
ORDER BY
    order_date;


## Limpieza opcional de widgets

Si necesitas eliminar los widgets al final de la demo, puedes ejecutar estas sentencias una a una:

```sql
REMOVE WIDGET table_name;
REMOVE WIDGET selected_country;
REMOVE WIDGET selected_status;
REMOVE WIDGET minimum_amount;
```

No las ejecutes durante la demo si quieres seguir usando los filtros.


# Conclusión de la demo

En esta demo se ha usado Markdown para hacer explícita la intención de cada bloque del notebook.

Un notebook bien documentado debe permitir responder rápidamente:

| Pregunta | Respuesta esperada |
|---|---|
| ¿Qué tabla se usa? | `training.lesson_01.orders_delta_demo` |
| ¿Qué se valida? | Estructura, volumen y resultados agregados |
| ¿Qué lenguaje se usa? | SQL |
| ¿Qué aporta Markdown? | Contexto, trazabilidad y legibilidad |

La práctica de los alumnos deberá seguir este mismo patrón: documentar cada consulta antes de ejecutarla.


In [0]:
# 1
%sql
SELECT
    status,
    COUNT(*) AS total_orders
FROM training.lesson_01.orders_delta_pair_01
GROUP BY
    status
ORDER BY
    total_orders DESC;

In [0]:
# 2 
%sql
SELECT
    country,
    ROUND(SUM(amount), 2) AS total_amount
FROM training.lesson_01.orders_delta_pair_01
GROUP BY
    country
ORDER BY
    total_amount DESC;

In [0]:
# 3
%sql
SELECT
    country,
    ROUND(SUM(amount), 2) AS total_amount
FROM training.lesson_01.orders_delta_pair_01
GROUP BY
    country
ORDER BY
    total_amount DESC
LIMIT 1;

In [0]:
# 4
%sql
SELECT
    channel,
    ROUND(AVG(amount), 2) AS average_order_amount
FROM training.lesson_01.orders_delta_pair_01
GROUP BY
    channel
ORDER BY
    average_order_amount DESC;

In [0]:
# 5
%sql
SELECT
    country,
    COUNT(*) AS cancelled_orders
FROM training.lesson_01.orders_delta_pair_01
WHERE status = 'Cancelled'
GROUP BY
    country
ORDER BY
    cancelled_orders DESC;

In [0]:
# 6
%sql
SELECT
    order_date,
    COUNT(*) AS total_orders,
    ROUND(SUM(amount), 2) AS total_amount
FROM training.lesson_01.orders_delta_pair_01
GROUP BY
    order_date
ORDER BY
    order_date;

In [0]:
# 7
%sql
SELECT
    channel,
    COUNT(*) AS total_orders,
    ROUND(SUM(amount), 2) AS total_amount
FROM training.lesson_01.orders_delta_pair_01
GROUP BY
    channel
HAVING COUNT(*) > 10
ORDER BY
    total_orders DESC;

In [0]:
# 8
%sql
SELECT
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
    SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS null_order_date,
    SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) AS null_country,
    SUM(CASE WHEN channel IS NULL THEN 1 ELSE 0 END) AS null_channel,
    SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) AS null_amount,
    SUM(CASE WHEN status IS NULL THEN 1 ELSE 0 END) AS null_status
FROM training.lesson_01.orders_delta_pair_01;

In [0]:
%sql
CREATE WIDGET TEXT table_name DEFAULT "training.lesson_01.orders_delta_pair_01";


In [0]:
%sql
CREATE WIDGET DROPDOWN selected_country DEFAULT "Spain" CHOICES SELECT * FROM (
    VALUES
        ("Spain"),
        ("France"),
        ("Germany"),
        ("Italy"),
        ("Netherlands"),
        ("Portugal")
);


In [0]:
%sql
CREATE WIDGET DROPDOWN selected_status DEFAULT "Completed" CHOICES SELECT * FROM (
    VALUES
        ("Completed"),
        ("Cancelled"),
        ("Pending")
);


In [0]:
%sql
CREATE WIDGET TEXT minimum_amount DEFAULT "0";


In [0]:
# 9
%sql
SELECT
    :selected_country AS selected_country,
    :selected_status AS selected_status,
    CAST(:minimum_amount AS DOUBLE) AS minimum_amount,
    COUNT(*) AS total_orders,
    ROUND(SUM(amount), 2) AS total_amount
FROM IDENTIFIER(:table_name)
WHERE
    country = :selected_country
    AND status = :selected_status
    AND amount >= CAST(:minimum_amount AS DOUBLE);


In [0]:
%sql
SELECT
    channel,
    COUNT(*) AS total_orders,
    ROUND(SUM(amount), 2) AS total_amount,
    ROUND(AVG(amount), 2) AS average_order_amount
FROM IDENTIFIER(:table_name)
WHERE
    country = :selected_country
GROUP BY
    channel
ORDER BY
    total_amount DESC;


In [0]:
%sql
SELECT
    status,
    COUNT(*) AS total_orders,
    ROUND(SUM(amount), 2) AS total_amount,
    ROUND(AVG(amount), 2) AS average_order_amount
FROM IDENTIFIER(:table_name)
WHERE
    amount >= CAST(:minimum_amount AS DOUBLE)
GROUP BY
    status
ORDER BY
    total_orders DESC;


In [0]:
%sql
SELECT
    order_date,
    country,
    status,
    COUNT(*) AS total_orders,
    ROUND(SUM(amount), 2) AS total_amount
FROM IDENTIFIER(:table_name)
WHERE
    country = :selected_country
    AND status = :selected_status
    AND amount >= CAST(:minimum_amount AS DOUBLE)
GROUP BY
    order_date,
    country,
    status
ORDER BY
    order_date;
